# MMHC — Working with Custom Data

This notebook shows how to:
1. Bring your own CSV/Excel data
2. Handle different data types (categorical, string, integer)
3. Tune algorithm parameters
4. Compare different configurations
5. Export results

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mmhc import MMHC, MMHCConfig, mmhc, plot_dag
from mmhc.graph_utils import adjacency_to_networkx, adjacency_to_edges, is_dag

## 1. Loading Custom Data

MMHC works with **discrete / categorical** data. Each column should contain a small number of distinct values.

### From CSV
```python
data = pd.read_csv("your_data.csv")
```

### From Excel
```python
data = pd.read_excel("your_data.xlsx", sheet_name="Sheet1")
```

Below we create a synthetic example with mixed types to demonstrate preprocessing.

In [ ]:
# Simulate a dataset with string categories
rng = np.random.default_rng(42)
n = 3000

weather = rng.choice(["sunny", "cloudy", "rainy"], size=n, p=[0.5, 0.3, 0.2])
temperature = rng.choice(["cold", "warm", "hot"], size=n, p=[0.3, 0.4, 0.3])

# ice_cream depends on weather and temperature
ice_cream = []
for w, t in zip(weather, temperature):
    if w == "sunny" and t == "hot":
        ice_cream.append(rng.choice(["yes", "no"], p=[0.9, 0.1]))
    elif w == "rainy":
        ice_cream.append(rng.choice(["yes", "no"], p=[0.1, 0.9]))
    else:
        ice_cream.append(rng.choice(["yes", "no"], p=[0.5, 0.5]))

# park_visit depends on weather
park_visit = []
for w in weather:
    if w == "sunny":
        park_visit.append(rng.choice(["yes", "no"], p=[0.8, 0.2]))
    else:
        park_visit.append(rng.choice(["yes", "no"], p=[0.3, 0.7]))

custom_data = pd.DataFrame({
    "weather": weather,
    "temperature": temperature,
    "ice_cream": ice_cream,
    "park_visit": park_visit,
})

custom_data.head(10)

In [ ]:
# MMHC handles string columns automatically (auto-encodes to 0-indexed integers)
result = mmhc(custom_data, config=MMHCConfig(random_seed=42))

print(f"Score: {result.score:.2f}")
print(f"\nEdges:")
for i, src in enumerate(result.column_names):
    for j, dst in enumerate(result.column_names):
        if result.adjacency_matrix[i, j] == 1:
            print(f"  {src} -> {dst}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
plot_dag(result.adjacency_matrix, result.column_names, ax=ax, title="Custom Data Network")
plt.tight_layout()
plt.show()

## 2. Tuning Parameters

| Parameter | Effect when increased | Effect when decreased |
|-----------|----------------------|----------------------|
| `alpha` | More edges (less strict independence test) | Fewer edges (stricter) |
| `eta` | Stronger prior, favours simpler graphs | Weaker prior, more data-driven |
| `max_iterations` | More search time | Faster, may miss edges |
| `early_stop_rounds` | More patience before stopping | Stops sooner |

In [ ]:
from mmhc import make_student

data = make_student(5000, random_state=42)

configs = {
    "strict (alpha=0.01)": MMHCConfig(alpha=0.01, random_seed=42),
    "default (alpha=0.05)": MMHCConfig(alpha=0.05, random_seed=42),
    "lenient (alpha=0.10)": MMHCConfig(alpha=0.10, random_seed=42),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, config) in zip(axes, configs.items()):
    r = mmhc(data, config=config)
    n_edges = int(r.adjacency_matrix.sum())
    plot_dag(r.adjacency_matrix, r.column_names, ax=ax, title=f"{name}\n{n_edges} edges, score={r.score:.0f}")

plt.tight_layout()
plt.show()

## 3. Comparing Eta Values

In [ ]:
etas = [0.1, 1.0, 10.0]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, eta in zip(axes, etas):
    r = mmhc(data, config=MMHCConfig(eta=eta, random_seed=42))
    n_edges = int(r.adjacency_matrix.sum())
    plot_dag(r.adjacency_matrix, r.column_names, ax=ax, title=f"eta={eta}\n{n_edges} edges, score={r.score:.0f}")

plt.tight_layout()
plt.show()

## 4. Exporting Results

In [ ]:
# Export adjacency matrix as CSV
result = mmhc(data, config=MMHCConfig(random_seed=42))

adj_df = pd.DataFrame(
    result.adjacency_matrix,
    index=result.column_names,
    columns=result.column_names,
)
adj_df.to_csv("adjacency_matrix.csv")
print("Saved adjacency_matrix.csv")

# Export edge list
edges = []
for i, src in enumerate(result.column_names):
    for j, dst in enumerate(result.column_names):
        if result.adjacency_matrix[i, j] == 1:
            edges.append({"source": src, "target": dst})

edge_df = pd.DataFrame(edges)
edge_df.to_csv("edge_list.csv", index=False)
print("Saved edge_list.csv")
edge_df

In [ ]:
# Export as NetworkX graph for further analysis
g = adjacency_to_networkx(result.adjacency_matrix, result.column_names)

# Example: topological ordering
import networkx as nx

print("Topological order:", list(nx.topological_sort(g)))
print("Is DAG:", nx.is_directed_acyclic_graph(g))

## 5. Working with NumPy Arrays

You can also pass a raw integer NumPy array directly (0-indexed values).

In [ ]:
# Raw numpy input
rng = np.random.default_rng(42)
x = rng.choice(3, size=2000)
y = (x + rng.choice(2, size=2000)) % 3  # y depends on x
z = rng.choice(2, size=2000)             # z is independent

raw_data = np.column_stack([x, y, z])
result = mmhc(raw_data, config=MMHCConfig(random_seed=42))

print(f"Edges (using numeric column names):")
for i in range(raw_data.shape[1]):
    for j in range(raw_data.shape[1]):
        if result.adjacency_matrix[i, j] == 1:
            print(f"  var_{i} -> var_{j}")